# Brax Ant — LWR-EGGROLL

**Architecture:** MLP [27, 256, 256, 256, 8] · POP=2048 · σ=0.2 · lr=0.1 · 300 gens · seeds [0,1,2]

### How to run
1. Run **Cell 1** once. It sets up the environment. **In Colab, restart the runtime afterwards** (Runtime → Restart runtime); in a local/uni Jupyter session no restart is needed.
2. Run **Cell 2** (all common code), then the **Pilot** cell — in that order.
3. Then run whichever method cell you want; each is self-contained and re-runnable.

### Two run modes (set in Cell 2)
- `RUN_MODE = "reuse"` *(default)* — loads the committed result JSONs: method cells skip, the pilot short-circuits on `pilot_results.json`. Fast way to confirm the notebook runs.
- `RUN_MODE = "fresh"` — regenerates everything from zero. Writes to a **separate** `results/brax_ant_rerun/` folder so the committed submission results are never overwritten. Runs the full pilot and trains every method (hours, needs a GPU).

### Where it runs (auto-detected in Cell 1)
- **Local / university environment:** open this notebook from inside the cloned repo. Paths resolve relative to the repo root; the base EGGROLL library is installed from its public pin.
- **Colab:** clone the repo locally with your own credentials, copy the repo folder into your Google Drive, and set `DRIVE_REPO_ROOT` in Cell 1 to point at it. No token is stored in this notebook.

Each method cell defines its allocation inline as `{(256,27): input, (256,256): hidden, (8,256): output}` — edit those three numbers to try a different allocation.

In [ ]:
# ── Cell 0 — First-run repo setup (Colab only) ──
# One-time: unpacks the repo you uploaded to Drive into MyDrive/gxs523.
#   1. Locally, build gxs523.zip (see SETTING_UP.md) and upload it to your
#      Google Drive via drive.google.com. Dropping it in My Drive root is
#      cleanest, but this cell will also find it in a subfolder.
#   2. Run this cell. It mounts Drive and, if MyDrive/gxs523 doesn't exist
#      yet, extracts the zip there. Re-running is safe (skips if present).
# Outside Colab (local/uni Jupyter) this cell does nothing.

import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    MYDRIVE   = Path("/content/drive/MyDrive")
    REPO_DEST = MYDRIVE / "gxs523"

    if REPO_DEST.exists() and (REPO_DEST / "pyproject.toml").exists():
        print(f"Repo already present at {REPO_DEST} — nothing to do.")
    else:
        # Prefer My Drive root; otherwise search the rest of My Drive.
        zip_path = MYDRIVE / "gxs523.zip"
        if not zip_path.exists():
            print("gxs523.zip not in My Drive root — searching your Drive...")
            matches = sorted(MYDRIVE.glob("**/gxs523.zip"))
            if matches:
                zip_path = matches[0]
                print(f"  found: {zip_path}")
            else:
                raise FileNotFoundError(
                    "gxs523.zip not found anywhere in My Drive. Upload it via "
                    "drive.google.com (My Drive root is easiest), then re-run "
                    "this cell.")

        import zipfile
        print(f"Extracting {zip_path} -> {REPO_DEST} ...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(MYDRIVE)   # zip contains a top-level gxs523/ folder

        if not (REPO_DEST / "pyproject.toml").exists():
            raise RuntimeError(
                "Extracted, but no pyproject.toml at MyDrive/gxs523. The zip "
                "must contain a top-level folder named 'gxs523' (see the zip "
                "command in SETTING_UP.md).")
        print(f"Done. Repo ready at {REPO_DEST}")
else:
    print("Not in Colab — skipping. Cell 1 will find the repo on disk.")

In [ ]:
# ── Cell 1 — Environment setup ──
# Works in two environments, auto-detected:
#   • Local / university Jupyter: run this notebook from inside the cloned repo.
#   • Google Colab: clone the repo locally, copy it into YOUR Google Drive,
#     then set DRIVE_REPO_ROOT below. (No token is stored in this notebook.)
#
# JAX is pinned to 0.9.0.1 to match the shipped Brax Ant results.

import os, sys, shutil, subprocess
from pathlib import Path

# Colab users: point this at the repo folder you copied into your Drive.
DRIVE_REPO_ROOT = "/content/drive/MyDrive/gxs523"

# Base EGGROLL library (public, pinned to the exact commit used for the results)
HYPERSCALEES_PIN = ("git+https://github.com/ESHyperscale/HyperscaleES.git"
                    "@b77f7d6f91238fd575313e946b9cad21e0a74b32")

def _find_repo_root(start: Path):
    """Walk upward from `start` looking for the repo root (pyproject.toml)."""
    for d in [start] + list(start.parents):
        if (d / "pyproject.toml").exists() and (d / "lwr_eggroll").exists():
            return d
    return None

# Detect Colab
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = Path(DRIVE_REPO_ROOT)
    if not (REPO_ROOT / "pyproject.toml").exists():
        raise FileNotFoundError(
            f"No repo at DRIVE_REPO_ROOT={DRIVE_REPO_ROOT}. Clone the repo "
            "locally, copy the whole folder into your Google Drive, and set "
            "DRIVE_REPO_ROOT to that path.")
else:
    REPO_ROOT = _find_repo_root(Path.cwd())
    if REPO_ROOT is None:
        raise FileNotFoundError(
            "Could not find the repo root (a folder with pyproject.toml and "
            "lwr_eggroll/). Open this notebook from inside the cloned repo.")

print(f"Repo root: {REPO_ROOT}")

# ── Installs ──
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "jax[cuda12]==0.9.0.1", "brax==0.10.5", "cloudpickle", "flax", "optax"])

# Base EGGROLL library (provides `hyperscalees`)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    HYPERSCALEES_PIN])

# This repo's package (provides lwr_eggroll pilot/selector modules)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e",
    str(REPO_ROOT)])

# Copy the LWR-EGGROLL noiser into the installed hyperscalees/noiser/ dir,
# so `from hyperscalees.noiser.lwr_eggroll import LWREggRoll` resolves.
import hyperscalees as _hs
_noiser_dir = Path(_hs.__file__).parent / "noiser"
shutil.copy2(REPO_ROOT / "experiments" / "lwr_eggroll.py",
             _noiser_dir / "lwr_eggroll.py")
print(f"Copied lwr_eggroll.py -> {_noiser_dir/'lwr_eggroll.py'}")

# XLA flags
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_cublaslt=false"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

# Persist REPO_ROOT to a file so it survives a runtime restart
# (environment variables set from Python are wiped on restart).
os.environ["LWR_REPO_ROOT"] = str(REPO_ROOT)
_marker = Path("/content/lwr_repo_root.txt") if IN_COLAB else (REPO_ROOT / ".lwr_repo_root.txt")
try:
    _marker.write_text(str(REPO_ROOT))
except Exception:
    pass

if IN_COLAB:
    print("Environment ready.")
    print("If Cell 2 later errors with a JAX/CUDA version conflict, restart the "
          "runtime (Runtime -> Restart) and run THIS cell (Cell 1) again before "
          "Cell 2 — a restart clears the environment, and re-running Cell 1 "
          "restores it. Otherwise just continue to Cell 2 (no restart needed).")
else:
    print("Environment ready. Run Cell 2 onward (no restart needed locally).")

In [ ]:
"""Brax Ant — LWR-EGGROLL common code (config, I/O, train, pilot, run_experiment).

Same codebase as the Brax 2 runs. Vmapped population evaluation with OOM
chunk-halving. Run this whole cell once (after restarting the runtime), then
run the Pilot cell, then any method cell.
"""
import os

import ast

def _safe_wrap_key(leaf):
    """Convert legacy PRNG keys to typed; skip if already typed."""
    if hasattr(leaf, "ndim") and leaf.ndim >= 1:
        return jax.random.wrap_key_data(leaf)
    return leaf
import json
import pickle
import cloudpickle
import sys
import time
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import optax

from brax import envs

import hyperscalees as hs
from hyperscalees.noiser.eggroll import EggRoll
from hyperscalees.noiser.lwr_eggroll import LWREggRoll

# ── Configuration ───────────────────────────────────────────────────
ENV_NAME = "ant"
EPISODE_LENGTH = 1000

POP_SIZE = 2048
SIGMA = 0.2
LR = 0.1
SIGMA_DECAY = 0.999
LR_DECAY = 0.9995
RANK = 4
N_LAYERS = 3
LAYER_SIZE = 256
ACTIVATION = "pqn"
OPTIMIZER = optax.sgd

# vmap chunk size — lower this if GPU OOM (e.g. 512, 256)
VMAP_CHUNK = 2056  # Usable for A100; auto-halves on OOM

MAX_GENS = 300
MAX_GENS_FLOOR = 300
SEEDS = [0,1,2]  # Single seed for this test

PHASE1_CHECKPOINT_GENS = 25
PHASE1_ELEVATION_GENS = 10
PHASE2_GENS = 15
PHASE3_GENS = 15
PILOT_SEEDS = [0, 1, 2]

NAN_REPLACEMENT = -1000.0
CHECKPOINT_INTERVAL_S = 600

# ── Run mode ──────────────────────────────────────────────────────
# "reuse" : load the committed result JSONs (method cells skip, pilot
#           short-circuits on pilot_results.json). Fast check that the
#           notebook runs.
# "fresh" : regenerate everything from zero, written to a SEPARATE folder
#           (brax_ant_rerun/) so the committed submission results are never
#           overwritten. Full pilot + trains every method (hours, GPU).
RUN_MODE = "reuse"
assert RUN_MODE in ("reuse", "fresh"), "RUN_MODE must be 'reuse' or 'fresh'"

# Repo root comes from Cell 1's environment detection.
# Resolve REPO_ROOT: env var first, then the marker file Cell 1 wrote
# (env vars don't survive a runtime restart; the file does).
_rr = os.environ.get("LWR_REPO_ROOT")
if not _rr:
    for _m in [Path("/content/lwr_repo_root.txt"), Path(".lwr_repo_root.txt")]:
        if _m.exists():
            _rr = _m.read_text().strip()
            break
if not _rr:
    raise RuntimeError(
        "REPO_ROOT not set. Run Cell 1 first (it detects the repo and records "
        "its path). After a runtime restart, re-run Cell 1 before this cell.")
REPO_ROOT = Path(_rr)
_RESULTS_BASE = REPO_ROOT / "results"

if RUN_MODE == "reuse":
    RESULTS_DIR = _RESULTS_BASE / "brax_ant"
else:  # fresh
    RESULTS_DIR = _RESULTS_BASE / "brax_ant_rerun"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RESULTS_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_MODE={RUN_MODE}  ->  RESULTS_DIR={RESULTS_DIR}")

LAYER_SHAPES = {
    "input":  (256, 27),
    "hidden": (256, 256),
    "output": (8, 256),
}

MODEL = hs.models.common.MLP

# ── LWR 4-2-0 allocation ──────────────────────────────────────────
# input=4 (most sensitive), hidden=2, output=0 (frozen)
LWR_420_SPEC = {
    (256, 27):  4,   # input
    (256, 256): 2,   # hidden
    (8, 256):   1,   # output — frozen
}
LWR_840_SPEC = {
      (256, 27):  8,   # input
      (256, 256): 4,   # hidden
      (8, 256):   1,   # output — frozen
}
# ── Environment ────────────────────────────────────────────────────
def make_env():
    return envs.get_environment(ENV_NAME)

def get_dims():
    env = make_env()
    return env.observation_size, env.action_size

OBS_DIM, ACT_DIM = get_dims()

# ── Checkpoint I/O (numpy-safe for cross-session pickle) ──────────
def _to_numpy_leaf(x):
    if hasattr(x, 'dtype') and hasattr(x, 'shape') and not isinstance(x, np.ndarray):
        try:
            return np.asarray(x)
        except Exception:
            return x
    return x

def _to_jax_leaf(x):
    if isinstance(x, np.ndarray):
        return jnp.asarray(x)
    return x

def save_pickle(filepath, data):
    filepath = Path(filepath)
    np_data = jax.tree_util.tree_map(_to_numpy_leaf, data)
    tmp = filepath.with_suffix('.tmp')
    with open(tmp, 'wb') as f:
        cloudpickle.dump(np_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    tmp.rename(filepath)

def load_pickle(filepath):
    filepath = Path(filepath)
    if not filepath.exists():
        return None
    try:
        with open(filepath, 'rb') as f:
            np_data = cloudpickle.load(f)
        return jax.tree_util.tree_map(_to_jax_leaf, np_data)
    except Exception as e:
        print(f"  WARNING: corrupt checkpoint {filepath}, ignoring: {e}", flush=True)
        return None

def save_json(filepath, data):
    filepath = Path(filepath)
    tmp = filepath.with_suffix('.tmp')
    with open(tmp, 'w') as f:
        json.dump(data, f, indent=2)
    tmp.rename(filepath)

def load_json(filepath):
    filepath = Path(filepath)
    if not filepath.exists():
        return None
    try:
        with open(filepath) as f:
            return json.load(f)
    except Exception:
        return None

# ── Model + Noiser Init ───────────────────────────────────────────
def init_model(key):
    fp, p, sm, em = MODEL.rand_init(
        key, in_dim=OBS_DIM, out_dim=ACT_DIM,
        hidden_dims=[LAYER_SIZE] * N_LAYERS,
        use_bias=True, activation=ACTIVATION, dtype="float32")
    return fp, p, sm, em

def init_noiser(params, noiser_class, rank_spec, sigma=SIGMA, lr=LR):
    fnp, np_ = noiser_class.init_noiser(
        params, sigma, lr, solver=OPTIMIZER, solver_kwargs={}, rank=rank_spec)
    return fnp, np_

# ── Training Loop (VMAPPED population evaluation) ─────────────────
def train(seed, noiser_class, rank_spec, label, max_gens=MAX_GENS,
          sigma=SIGMA, lr=LR, sigma_decay=SIGMA_DECAY, lr_decay=LR_DECAY,
          initial_state=None, return_checkpoint_at=None):
    NOISER = noiser_class
    env = make_env()

    ckpt_file = CACHE_DIR / f"train_{label}_seed{seed}.pkl"
    resumed_gen = 0
    history = []
    best_fitness = -float("inf")

    existing_ckpt = load_pickle(ckpt_file)
    if existing_ckpt is not None and initial_state is None:
        resumed_gen = existing_ckpt['gen'] + 1
        history = existing_ckpt['history']
        best_fitness = existing_ckpt['best_fitness']
        frozen_params = existing_ckpt['frozen_params']
        current_params = existing_ckpt['params']
        scan_map = existing_ckpt['scan_map']
        es_map = existing_ckpt['es_map']
        es_tree_key = existing_ckpt['es_tree_key']
        es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
        es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
        es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
        es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
        es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
        es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
        frozen_noiser_params = existing_ckpt['frozen_noiser_params']
        current_noiser_params = existing_ckpt['noiser_params']
        episode_key = existing_ckpt['episode_key']
        print(f"\n  [{label} seed={seed}] RESUMING from gen {resumed_gen} "
              f"(best={best_fitness:.1f})", flush=True)
    else:
        if initial_state is not None:
            frozen_params = initial_state['frozen_params']
            current_params = initial_state['params']
            scan_map = initial_state['scan_map']
            es_map = initial_state['es_map']
            es_tree_key = initial_state['es_tree_key']
            es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
            es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
            es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
            es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
            es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
            es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
            frozen_noiser_params, current_noiser_params = init_noiser(
                current_params, noiser_class, rank_spec, sigma=sigma, lr=lr)
        else:
            key = jax.random.PRNGKey(seed)
            key, model_key, es_key = jax.random.split(key, 3)
            frozen_params, current_params, scan_map, es_map = init_model(model_key)
            es_tree_key = hs.models.common.simple_es_tree_key(
                current_params, es_key, scan_map)
            es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
            es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
            es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
            es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
            es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
            es_tree_key = jax.tree.map(_safe_wrap_key, es_tree_key)
            frozen_noiser_params, current_noiser_params = init_noiser(
                current_params, noiser_class, rank_spec, sigma=sigma, lr=lr)
        episode_key = jax.random.PRNGKey(seed + 10000)
        print(f"\n  [{label} seed={seed}] initialising...", flush=True)

    n_params = sum(x.size for x in jax.tree_util.tree_leaves(current_params))
    print(f"  [{label} seed={seed}] params: {n_params:,}", flush=True)

    all_mids = jnp.arange(POP_SIZE, dtype=jnp.int32)

    checkpoint_state_for_return = None
    t0 = time.time()
    last_ckpt_time = time.time()

    def _make_eval_chunk(fnp, fp, estk):
        def _eval_single(gen_i, mid_i, ep_key, cnp, cp):
            iterinfo = (gen_i, mid_i)
            def policy(obs):
                return jnp.clip(
                    MODEL.forward(NOISER, fnp, cnp, fp, cp, estk, iterinfo, obs),
                    -1.0, 1.0)
            state = env.reset(ep_key)
            def scan_step(carry, _):
                st, total_reward, done = carry
                action = policy(st.obs)
                ns = env.step(st, action)
                reward = ns.reward * (1.0 - done)
                done = jnp.logical_or(done, ns.done)
                return (ns, total_reward + reward, done), None
            (_, total_reward, _), _ = jax.lax.scan(
                scan_step, (state, jnp.float32(0.0), jnp.bool_(False)),
                None, length=EPISODE_LENGTH)
            return total_reward
        return jax.jit(jax.vmap(
            _eval_single,
            in_axes=(None, 0, 0, None, None)
        ))

    _eval_chunk_vmapped = _make_eval_chunk(
        frozen_noiser_params, frozen_params, es_tree_key)

    def _make_eval_single_jit(fnp, fp, estk):
        @jax.jit
        def _eval_one(gen_i, mid_i, ep_key, cnp, cp):
            iterinfo = (gen_i, mid_i)
            def policy(obs):
                return jnp.clip(
                    MODEL.forward(NOISER, fnp, cnp, fp, cp, estk, iterinfo, obs),
                    -1.0, 1.0)
            state = env.reset(ep_key)
            def scan_step(carry, _):
                st, total_reward, done = carry
                action = policy(st.obs)
                ns = env.step(st, action)
                reward = ns.reward * (1.0 - done)
                done = jnp.logical_or(done, ns.done)
                return (ns, total_reward + reward, done), None
            (_, total_reward, _), _ = jax.lax.scan(
                scan_step, (state, jnp.float32(0.0), jnp.bool_(False)),
                None, length=EPISODE_LENGTH)
            return total_reward
        return _eval_one

    _eval_one_jit = _make_eval_single_jit(
        frozen_noiser_params, frozen_params, es_tree_key)

    _use_vmap = True
    _active_chunk = VMAP_CHUNK

    def eval_population_vmap(gen_i, all_mids, all_keys, cnp, cp, chunk):
        chunks = []
        for start in range(0, POP_SIZE, chunk):
            end = min(start + chunk, POP_SIZE)
            chunk_rewards = _eval_chunk_vmapped(
                gen_i, all_mids[start:end], all_keys[start:end], cnp, cp)
            chunks.append(chunk_rewards)
        return jnp.concatenate(chunks)

    def eval_population_sequential(gen_i, all_keys, cnp, cp):
        rewards = []
        for mid in range(POP_SIZE):
            r = _eval_one_jit(gen_i, jnp.int32(mid), all_keys[mid], cnp, cp)
            rewards.append(r)
        return jnp.stack(rewards)

    for gen in range(resumed_gen, max_gens):
        t_gen = time.time()

        all_keys = jax.random.split(episode_key, POP_SIZE + 1)
        episode_key = all_keys[0]
        member_keys = all_keys[1:]

        gen_i = jnp.int32(gen)

        if _use_vmap:
            try:
                gen_fitnesses_arr = eval_population_vmap(
                    gen_i, all_mids, member_keys,
                    current_noiser_params, current_params, _active_chunk)
            except Exception as e:
                err_msg = str(e).lower()
                if "out of memory" in err_msg or "resource_exhausted" in err_msg:
                    _active_chunk = max(1, _active_chunk // 2)
                    print(f"  \u26a0 vmap OOM at chunk {_active_chunk * 2}, "
                          f"retrying with chunk {_active_chunk}...", flush=True)
                    if _active_chunk < 8:
                        print(f"  \u26a0 vmap chunk too small, falling back to "
                              f"sequential (slower but safe)", flush=True)
                        _use_vmap = False
                        gen_fitnesses_arr = eval_population_sequential(
                            gen_i, member_keys,
                            current_noiser_params, current_params)
                    else:
                        try:
                            gen_fitnesses_arr = eval_population_vmap(
                                gen_i, all_mids, member_keys,
                                current_noiser_params, current_params,
                                _active_chunk)
                        except Exception:
                            print(f"  \u26a0 vmap failed again, falling back to "
                                  f"sequential", flush=True)
                            _use_vmap = False
                            gen_fitnesses_arr = eval_population_sequential(
                                gen_i, member_keys,
                                current_noiser_params, current_params)
                else:
                    raise
        else:
            gen_fitnesses_arr = eval_population_sequential(
                gen_i, member_keys,
                current_noiser_params, current_params)

        # NaN handling
        nan_mask = jnp.isnan(gen_fitnesses_arr)
        nan_count = int(jnp.sum(nan_mask))
        if nan_count > 0:
            print(f"  \u26a0 gen {gen}: {nan_count}/{POP_SIZE} NaN fitnesses, "
                  f"replacing with {NAN_REPLACEMENT}", flush=True)
            gen_fitnesses_arr = jnp.where(nan_mask, NAN_REPLACEMENT, gen_fitnesses_arr)

        gen_mean = float(jnp.mean(gen_fitnesses_arr))
        gen_best = float(jnp.max(gen_fitnesses_arr))
        gen_var = float(jnp.var(gen_fitnesses_arr))
        if gen_best > best_fitness:
            best_fitness = gen_best

        history.append({
            "gen": gen, "mean_fitness": gen_mean, "best_fitness": gen_best,
            "best_so_far": best_fitness, "fitness_variance": gen_var,
            "nan_count": nan_count, "wall_s": time.time() - t0,
        })

        # ES update
        iterinfo = (jnp.full(POP_SIZE, gen, dtype=jnp.int32),
                    jnp.arange(POP_SIZE))
        converted = NOISER.convert_fitnesses(
            frozen_noiser_params, current_noiser_params, gen_fitnesses_arr)
        current_noiser_params, current_params = NOISER.do_updates(
            frozen_noiser_params, current_noiser_params, current_params,
            es_tree_key, converted, iterinfo, es_map)

        # Decay
        if sigma_decay < 1.0:
            current_noiser_params['sigma'] = current_noiser_params['sigma'] * sigma_decay
        if lr_decay < 1.0 and 'lr' in current_noiser_params:
            current_noiser_params['lr'] = current_noiser_params['lr'] * lr_decay

        # Return checkpoint (Phase 1)
        if return_checkpoint_at is not None and gen == return_checkpoint_at:
            checkpoint_state_for_return = {
                'frozen_params': frozen_params, 'params': current_params,
                'scan_map': scan_map, 'es_map': es_map,
                'es_tree_key': es_tree_key,
                'frozen_noiser_params': frozen_noiser_params,
                'noiser_params': current_noiser_params,
            }
            print(f"  [{label} seed={seed}] return-checkpoint at gen {gen}", flush=True)

        # Crash-recovery checkpoint every 10 min
        now = time.time()
        if now - last_ckpt_time >= CHECKPOINT_INTERVAL_S:
            save_pickle(ckpt_file, {
                'gen': gen, 'history': history, 'best_fitness': best_fitness,
                'frozen_params': frozen_params, 'params': current_params,
                'scan_map': scan_map, 'es_map': es_map,
                'es_tree_key': es_tree_key,
                'frozen_noiser_params': frozen_noiser_params,
                'noiser_params': current_noiser_params,
                'episode_key': episode_key,
            })
            last_ckpt_time = now
            print(f"  [{label} seed={seed}] checkpoint at gen {gen} "
                  f"({now - t0:.0f}s elapsed)", flush=True)

        gen_elapsed = time.time() - t_gen
        if gen % 5 == 0 or gen == max_gens - 1:
            print(f"    gen {gen:4d}  mean={gen_mean:.1f}  best_so_far={best_fitness:.1f}"
                  f"  sigma={float(current_noiser_params['sigma']):.4f}"
                  f"  ({gen_elapsed:.1f}s)", flush=True)

    total_time = time.time() - t0
    final_mean = history[-1]["mean_fitness"] if history else 0.0
    print(f"  [{label} seed={seed}] done {total_time:.0f}s, best={best_fitness:.1f}",
          flush=True)

    if ckpt_file.exists():
        pass  # KEEP checkpoints

    result = {
        "method": label, "seed": seed, "best_fitness": best_fitness,
        "final_mean_fitness": final_mean, "generations": len(history),
        "wall_seconds": total_time, "history": history,
    }
    if return_checkpoint_at is not None:
        return result, checkpoint_state_for_return
    return result


# ── Phase 1: Elevation ────────────────────────────────────────────
def run_phase1():
    print(f"\n{'='*60}")
    print("SENSITIVITY PILOT — Phase 1 (Elevation / Magnitude)")
    print(f"{'='*60}")
    print(f"  Checkpoint gens: {PHASE1_CHECKPOINT_GENS}, "
          f"Elevation gens: {PHASE1_ELEVATION_GENS}, Seeds: {PILOT_SEEDS}")

    unique_shapes = list(LAYER_SHAPES.items())
    phase1_results = {}

    for seed in PILOT_SEEDS:
        ckpt_cache = CACHE_DIR / f"p1_checkpoint_seed{seed}.json"
        ckpt_state_file = CACHE_DIR / f"p1_checkpoint_state_seed{seed}.pkl"
        cached = load_json(ckpt_cache)

        if cached is not None and load_pickle(ckpt_state_file) is not None:
            checkpoint_fitness = cached["checkpoint_fitness"]
            checkpoint_state = load_pickle(ckpt_state_file)
            print(f"\n  Phase 1 — Checkpoint seed={seed} from cache "
                  f"(fitness={checkpoint_fitness:.1f})", flush=True)
        else:
            print(f"\n  Phase 1 — Training shared checkpoint (seed={seed})...",
                  flush=True)
            checkpoint_result, checkpoint_state = train(
                seed, EggRoll, RANK, f"phase1_checkpoint",
                max_gens=PHASE1_CHECKPOINT_GENS,
                return_checkpoint_at=PHASE1_CHECKPOINT_GENS - 1)
            checkpoint_fitness = checkpoint_result["final_mean_fitness"]
            save_json(ckpt_cache, {"checkpoint_fitness": checkpoint_fitness})
            save_pickle(ckpt_state_file, checkpoint_state)
            print(f"  Checkpoint fitness (seed={seed}): {checkpoint_fitness:.1f}",
                  flush=True)

        if checkpoint_state is None:
            print(f"  ERROR: checkpoint not captured for seed={seed}!", flush=True)
            continue

        for layer_name, layer_shape in unique_shapes:
            elev_cache = CACHE_DIR / f"p1_elevate_{layer_name}_seed{seed}.json"
            cached_elev = load_json(elev_cache)
            if cached_elev is not None:
                phase1_results[f"{layer_name}_seed{seed}"] = cached_elev
                print(f"  Phase 1 — {layer_name} seed={seed} from cache "
                      f"(|delta|={cached_elev['abs_delta']:.1f})", flush=True)
                continue

            print(f"\n  Phase 1 — Elevating {layer_name} {layer_shape} to r=8 "
                  f"(seed={seed}):", flush=True)
            elevated_spec = {s: RANK for s in LAYER_SHAPES.values()}
            elevated_spec[layer_shape] = 8
            elevated_result = train(
                seed, LWREggRoll, elevated_spec,
                f"phase1_elevate_{layer_name}",
                max_gens=PHASE1_ELEVATION_GENS,
                initial_state=checkpoint_state)
            elevated_fitness = elevated_result["final_mean_fitness"]
            delta = abs(elevated_fitness - checkpoint_fitness)
            entry = {
                "layer": layer_name, "seed": seed,
                "checkpoint_fitness": checkpoint_fitness,
                "elevated_fitness": elevated_fitness, "abs_delta": delta,
            }
            phase1_results[f"{layer_name}_seed{seed}"] = entry
            save_json(elev_cache, entry)
            print(f"  {layer_name} (seed={seed}): elevated={elevated_fitness:.1f}, "
                  f"|delta|={delta:.1f}", flush=True)

    phase1_summary = {}
    for layer_name, _ in unique_shapes:
        deltas = [phase1_results[f"{layer_name}_seed{s}"]["abs_delta"]
                  for s in PILOT_SEEDS
                  if f"{layer_name}_seed{s}" in phase1_results]
        phase1_summary[layer_name] = {
            "mean_abs_delta": float(np.mean(deltas)),
            "per_seed_deltas": deltas,
        }

    phase1_ordering = sorted(phase1_summary.keys(),
                             key=lambda n: phase1_summary[n]["mean_abs_delta"],
                             reverse=True)
    phase1_ordering_str = " > ".join(phase1_ordering)
    save_json(CACHE_DIR / "phase1_complete.json",
              {"ordering": phase1_ordering_str, "summary": phase1_summary})

    print(f"\n  Phase 1 ordering: {phase1_ordering_str}")
    for name in phase1_ordering:
        s = phase1_summary[name]
        print(f"    {name}: mean |delta| = {s['mean_abs_delta']:.1f}")
    return phase1_ordering_str, phase1_summary


# ── Phase 2: Causal Ablation ──────────────────────────────────────
def run_phase2():
    print(f"\n{'='*60}")
    print("SENSITIVITY PILOT — Phase 2 (Causal Ablation)")
    print(f"{'='*60}")
    print(f"  Gens: {PHASE2_GENS}, Seeds: {PILOT_SEEDS}, "
          f"Baseline: r={RANK}, Ablation: r=1")

    unique_shapes = list(LAYER_SHAPES.items())

    baseline_fitnesses = []
    baseline_bests = []
    for seed in PILOT_SEEDS:
        bl_cache = CACHE_DIR / f"p2_baseline_seed{seed}.json"
        cached = load_json(bl_cache)
        if cached is not None:
            baseline_fitnesses.append(cached["final_mean_fitness"])
            baseline_bests.append(cached.get("best_fitness", cached["final_mean_fitness"]))
            print(f"  Baseline seed={seed} from cache "
                  f"({cached['final_mean_fitness']:.1f})", flush=True)
        else:
            r = train(seed, EggRoll, RANK, f"pilot_baseline_r{RANK}",
                      max_gens=PHASE2_GENS)
            baseline_fitnesses.append(r["final_mean_fitness"])
            baseline_bests.append(r["best_fitness"])
            save_json(bl_cache, {"final_mean_fitness": r["final_mean_fitness"],
                                 "best_fitness": r["best_fitness"]})
    baseline_mean = float(np.mean(baseline_fitnesses))
    baseline_best = float(np.mean(baseline_bests))
    print(f"  Baseline mean: {baseline_mean:.1f}, best: {baseline_best:.1f}", flush=True)

    degradations = {}
    for layer_name, layer_shape in unique_shapes:
        abl_fitnesses = []
        abl_bests = []
        for seed in PILOT_SEEDS:
            abl_cache = CACHE_DIR / f"p2_ablate_{layer_name}_seed{seed}.json"
            cached = load_json(abl_cache)
            if cached is not None:
                abl_fitnesses.append(cached["final_mean_fitness"])
                abl_bests.append(cached.get("best_fitness", cached["final_mean_fitness"]))
                print(f"  Ablate {layer_name} seed={seed} from cache", flush=True)
            else:
                ablated_spec = {s: RANK for s in LAYER_SHAPES.values()}
                ablated_spec[layer_shape] = 1
                r = train(seed, LWREggRoll, ablated_spec,
                          f"pilot_ablate_{layer_name}", max_gens=PHASE2_GENS)
                abl_fitnesses.append(r["final_mean_fitness"])
                abl_bests.append(r["best_fitness"])
                save_json(abl_cache,
                          {"final_mean_fitness": r["final_mean_fitness"],
                           "best_fitness": r["best_fitness"]})
        ablated_mean = float(np.mean(abl_fitnesses))
        ablated_best = float(np.mean(abl_bests))
        degradation = baseline_mean - ablated_mean
        degradation_best = baseline_best - ablated_best
        degradations[layer_name] = {
            "shape": layer_shape,
            "mean_fitness": ablated_mean,
            "best_fitness": ablated_best,
            "degradation": degradation,
            "degradation_best": degradation_best,
        }
        print(f"  {layer_name}: mean={ablated_mean:.1f} (deg {degradation:.1f}), "
              f"best={ablated_best:.1f} (deg {degradation_best:.1f})", flush=True)

    # Mean-fitness ordering (primary for the mean_fitness experiment)
    ordering = sorted(degradations.keys(),
                      key=lambda n: degradations[n]["degradation"], reverse=True)
    ordering_str = " > ".join(ordering)
    # Best-fitness ordering (primary for the best_fitness experiment)
    ordering_best = sorted(degradations.keys(),
                           key=lambda n: degradations[n]["degradation_best"], reverse=True)
    ordering_best_str = " > ".join(ordering_best)

    save_json(CACHE_DIR / "phase2_complete.json", {
        "baseline_mean": baseline_mean,
        "baseline_best": baseline_best,
        "degradations": {k: {**v, "shape": str(v["shape"])}
                         for k, v in degradations.items()},
        "ordering": ordering_str,
        "ordering_best": ordering_best_str,
    })
    print(f"\n  Phase 2 mean-fitness ordering: {ordering_str}")
    print(f"  Phase 2 best-fitness ordering: {ordering_best_str}")
    return ordering, degradations, baseline_mean


# ── Phase 3: Binary Inclusion ─────────────────────────────────────
def run_phase3(ordering, degradations):
    least_sensitive = ordering[-1]
    ls_shape = degradations[least_sensitive]["shape"]
    print(f"\n{'='*60}")
    print(f"SENSITIVITY PILOT — Phase 3 (Binary Inclusion)")
    print(f"{'='*60}")
    print(f"  Layer: {least_sensitive} {ls_shape}, r=0 vs r=1")

    r1_mean = degradations[least_sensitive]["mean_fitness"]
    r1_best = degradations[least_sensitive].get("best_fitness", r1_mean)

    frozen_fitnesses = []
    frozen_bests = []
    for seed in PILOT_SEEDS:
        p3_cache = CACHE_DIR / f"p3_freeze_{least_sensitive}_seed{seed}.json"
        cached = load_json(p3_cache)
        if cached is not None:
            frozen_fitnesses.append(cached["final_mean_fitness"])
            frozen_bests.append(cached.get("best_fitness", cached["final_mean_fitness"]))
            print(f"  Freeze {least_sensitive} seed={seed} from cache", flush=True)
        else:
            frozen_spec = {s: RANK for s in LAYER_SHAPES.values()}
            frozen_spec[ls_shape] = 0
            r = train(seed, LWREggRoll, frozen_spec,
                      f"pilot_freeze_{least_sensitive}", max_gens=PHASE3_GENS)
            frozen_fitnesses.append(r["final_mean_fitness"])
            frozen_bests.append(r["best_fitness"])
            save_json(p3_cache, {"final_mean_fitness": r["final_mean_fitness"],
                                 "best_fitness": r["best_fitness"]})

    r0_mean = float(np.mean(frozen_fitnesses))
    r0_best = float(np.mean(frozen_bests))

    # Decision on mean_fitness (freeze if within 5.0 of rank 1)
    freeze_justified = r0_mean >= r1_mean - 5.0
    phase3_decision = 0 if freeze_justified else 1
    # Decision on best_fitness (freeze if rank-0 best >= rank-1 best)
    freeze_justified_best = r0_best >= r1_best
    phase3_decision_best = 0 if freeze_justified_best else 1

    save_json(CACHE_DIR / "phase3_complete.json", {
        "least_sensitive": least_sensitive,
        "r1_mean": r1_mean, "r0_mean": r0_mean,
        "r1_best": r1_best, "r0_best": r0_best,
        "freeze_justified": freeze_justified, "decision": phase3_decision,
        "freeze_justified_best": freeze_justified_best,
        "decision_best": phase3_decision_best,
    })
    print(f"  mean: r=1 {r1_mean:.1f}, r=0 {r0_mean:.1f} -> rank {phase3_decision}")
    print(f"  best: r=1 {r1_best:.1f}, r=0 {r0_best:.1f} -> rank {phase3_decision_best}")
    return least_sensitive, phase3_decision, r0_mean, r1_mean


# ── Full Pilot ─────────────────────────────────────────────────────
def run_sensitivity_pilot():
    # Short-circuit: if the cumulative pilot file exists, reuse it wholesale.
    pf = RESULTS_DIR / "pilot_results.json"
    cached_pilot = load_json(pf)
    if cached_pilot is not None:
        alloc_named = cached_pilot["allocation"]
        alloc_label = cached_pilot["allocation_label"]
        ordering_str = cached_pilot["phase2"]["ordering"]
        allocation = {LAYER_SHAPES[ln]: rv for ln, rv in alloc_named.items()}
        print(f"\n  Pilot from pilot_results.json: {alloc_named} "
              f"(label: {alloc_label})")
        return allocation, alloc_named, ordering_str, alloc_label

    # Phase 1
    p1c = load_json(CACHE_DIR / "phase1_complete.json")
    if p1c is not None:
        p1_ord_str, p1_summary = p1c["ordering"], p1c["summary"]
        print(f"\n  Phase 1 from cache: {p1_ord_str}")
    else:
        p1_ord_str, p1_summary = run_phase1()

    # Phase 2
    p2c = load_json(CACHE_DIR / "phase2_complete.json")
    if p2c is not None:
        ordering_str = p2c["ordering"]
        ordering = ordering_str.split(" > ")
        baseline_mean = p2c["baseline_mean"]
        degradations = {}
        for k, v in p2c["degradations"].items():
            shape = ast.literal_eval(v["shape"])
            degradations[k] = {**v, "shape": shape}
        print(f"\n  Phase 2 from cache: {ordering_str}")
    else:
        ordering, degradations, baseline_mean = run_phase2()
        ordering_str = " > ".join(ordering)

    ordering_best_str = " > ".join(
        sorted(degradations.keys(),
               key=lambda n: degradations[n].get("degradation_best",
                                                  degradations[n]["degradation"]),
               reverse=True))

    # Phase 3
    p3c = load_json(CACHE_DIR / "phase3_complete.json")
    if p3c is not None:
        least_sensitive = p3c["least_sensitive"]
        phase3_decision = p3c["decision"]
        r0_mean, r1_mean = p3c["r0_mean"], p3c["r1_mean"]
        print(f"\n  Phase 3 from cache: {least_sensitive} -> rank {phase3_decision}")
    else:
        least_sensitive, phase3_decision, r0_mean, r1_mean = \
            run_phase3(ordering, degradations)

    # Build mean-fitness allocation (rank 4 cap)
    rank_tiers = [4, 2]
    allocation = {}
    alloc_named = {}
    for i, layer_name in enumerate(ordering):
        shape = LAYER_SHAPES[layer_name]
        if i == len(ordering) - 1:
            allocation[shape] = phase3_decision
        else:
            allocation[shape] = rank_tiers[min(i, len(rank_tiers) - 1)]
        alloc_named[layer_name] = allocation[shape]

    alloc_label = "_".join(
        str(alloc_named[n]) for n in ["input", "hidden", "output"])

    save_json(RESULTS_DIR / "pilot_results.json", {
        "phase1": {"ordering": p1_ord_str, "summary": p1_summary},
        "phase2": {
            "baseline_mean": baseline_mean,
            "degradations": {k: {**v, "shape": str(v["shape"])}
                             for k, v in degradations.items()},
            "ordering": ordering_str,
            "ordering_best": ordering_best_str,
        },
        "phase3": {
            "least_sensitive": least_sensitive,
            "r1_mean": r1_mean, "r0_mean": r0_mean,
        },
        "allocation": alloc_named, "allocation_label": alloc_label,
        "rank_spec": {str(k): v for k, v in allocation.items()},
    })

    print(f"\n{'='*60}")
    print(f"  PILOT SUMMARY")
    print(f"  Phase 1 (magnitude):        {p1_ord_str}")
    print(f"  Phase 2 mean-fitness order: {ordering_str}")
    print(f"  Phase 2 best-fitness order: {ordering_best_str}")
    print(f"  Phase 3: {least_sensitive} -> rank {phase3_decision}")
    print(f"  Mean-fitness allocation: {alloc_named} (label: {alloc_label})")
    print(f"{'='*60}\n", flush=True)
    return allocation, alloc_named, ordering_str, alloc_label


# ── Per-experiment run helper ─────────────────────────────────────
def run_experiment(label, noiser_class, rank_spec, seeds=SEEDS,
                   max_gens=MAX_GENS):
    """Run one method across seeds. Skips seeds whose JSON already exists.
    rank_spec: int (uniform rank) for EggRoll, or a dict
    {(256,27): input, (256,256): hidden, (8,256): output} for LWREggRoll.
    Edit the three numbers in a method cell's spec to try a new allocation.
    """
    print("=" * 60)
    print(f"METHOD: {label}  ({max_gens} gens, seeds={seeds})")
    if isinstance(rank_spec, dict):
        named = {ln: rank_spec.get(sh) for ln, sh in LAYER_SHAPES.items()}
        print(f"  allocation (input,hidden,output) = "
              f"({named['input']},{named['hidden']},{named['output']})")
    else:
        print(f"  uniform rank = {rank_spec}")
    print("=" * 60, flush=True)

    method_results = []
    for seed in seeds:
        result_file = RESULTS_DIR / f"{label}_seed{seed}.json"
        if result_file.exists():
            r = load_json(result_file)
            if r is not None:
                print(f"  [{label} seed={seed}] exists "
                      f"(best={r.get('best_fitness', float('nan')):.1f}), skipping.",
                      flush=True)
                method_results.append(r)
                continue
        r = train(seed, noiser_class, rank_spec, label,
                  max_gens=max_gens, sigma_decay=SIGMA_DECAY, lr_decay=LR_DECAY)
        save_json(result_file, r)
        method_results.append(r)

    if method_results:
        bests = [r["best_fitness"] for r in method_results]
        finals = [r["final_mean_fitness"] for r in method_results]
        print(f"\n  {label}: mean_best={np.mean(bests):.1f}  "
              f"std_best={np.std(bests):.1f}  mean_final={np.mean(finals):.1f}")
        print(f"  per-seed best: {[round(b,1) for b in bests]}", flush=True)
    return method_results

print("Common code loaded. Run the Pilot cell next, then any experiment cell.")

# Pilot

In [ ]:
# reuse mode: short-circuits on the committed pilot_results.json.
# fresh mode: runs all three phases from scratch into brax_ant_rerun/.
alloc, alloc_named, ordering_str, alloc_label = run_sensitivity_pilot()

# Shared Baselines

#### EGGROLL rank 4

In [7]:
run_experiment("eggroll_r4", EggRoll, RANK)   # uniform rank 4

METHOD: eggroll_r4  (300 gens, seeds=[0, 1, 2])
  uniform rank = 4
  [eggroll_r4 seed=0] exists (best=7.6), skipping.
  [eggroll_r4 seed=1] exists (best=32.6), skipping.
  [eggroll_r4 seed=2] exists (best=41.5), skipping.

  eggroll_r4: mean_best=27.2  std_best=14.4  mean_final=-21.8
  per-seed best: [7.6, 32.6, 41.5]


[{'method': 'eggroll_r4',
  'seed': 0,
  'best_fitness': 7.600556373596191,
  'final_mean_fitness': -27.290924072265625,
  'generations': 300,
  'wall_seconds': 9931.05470943451,
  'history': [{'gen': 0,
    'mean_fitness': -122.32892608642578,
    'best_fitness': 5.378355979919434,
    'best_so_far': 5.378355979919434,
    'fitness_variance': 78943.9765625,
    'nan_count': 73,
    'wall_s': 65.4385175704956},
   {'gen': 1,
    'mean_fitness': -97.3203353881836,
    'best_fitness': -2.0813233852386475,
    'best_so_far': 5.378355979919434,
    'fitness_variance': 41428.3984375,
    'nan_count': 49,
    'wall_s': 107.51212215423584},
   {'gen': 2,
    'mean_fitness': -107.4285888671875,
    'best_fitness': -1.2929129600524902,
    'best_so_far': 5.378355979919434,
    'fitness_variance': 52627.9765625,
    'nan_count': 63,
    'wall_s': 140.48454546928406},
   {'gen': 3,
    'mean_fitness': -97.255615234375,
    'best_fitness': 5.550673484802246,
    'best_so_far': 5.550673484802246,
 

#### EGGROLL rank 1

In [8]:
run_experiment("eggroll_r1", EggRoll, 1)      # uniform rank 1

METHOD: eggroll_r1  (300 gens, seeds=[0, 1, 2])
  uniform rank = 1
  [eggroll_r1 seed=0] exists (best=94.7), skipping.
  [eggroll_r1 seed=1] exists (best=81.9), skipping.
  [eggroll_r1 seed=2] exists (best=53.9), skipping.

  eggroll_r1: mean_best=76.8  std_best=17.0  mean_final=-29.6
  per-seed best: [94.7, 81.9, 53.9]


[{'method': 'eggroll_r1',
  'seed': 0,
  'best_fitness': 94.73421478271484,
  'final_mean_fitness': -31.89275550842285,
  'generations': 300,
  'wall_seconds': 10132.670098543167,
  'history': [{'gen': 0,
    'mean_fitness': -193.0072021484375,
    'best_fitness': 94.73421478271484,
    'best_so_far': 94.73421478271484,
    'fitness_variance': 195337.25,
    'nan_count': 70,
    'wall_s': 58.25192165374756},
   {'gen': 1,
    'mean_fitness': -143.7296905517578,
    'best_fitness': 35.395751953125,
    'best_so_far': 94.73421478271484,
    'fitness_variance': 125911.109375,
    'nan_count': 35,
    'wall_s': 96.04660677909851},
   {'gen': 2,
    'mean_fitness': -112.86048126220703,
    'best_fitness': 26.36751365661621,
    'best_so_far': 94.73421478271484,
    'fitness_variance': 59290.94140625,
    'nan_count': 25,
    'wall_s': 129.76986980438232},
   {'gen': 3,
    'mean_fitness': -117.794921875,
    'best_fitness': 43.30618667602539,
    'best_so_far': 94.73421478271484,
    'fitne

# Metric: Mean Fitness

#### lwr_8_0_4

In [9]:
# lwr_8_0_4  (input=8, hidden=0, output=4)
# To try a different allocation, edit the three numbers below.
spec = {(256, 27): 8, (256, 256): 0, (8, 256): 4}
run_experiment("lwr_8_0_4", LWREggRoll, spec)

METHOD: lwr_8_0_4  (300 gens, seeds=[0, 1, 2])
  allocation (input,hidden,output) = (8,0,4)
  [lwr_8_0_4 seed=0] exists (best=11.1), skipping.
  [lwr_8_0_4 seed=1] exists (best=20.6), skipping.
  [lwr_8_0_4 seed=2] exists (best=18.6), skipping.

  lwr_8_0_4: mean_best=16.7  std_best=4.1  mean_final=-20.9
  per-seed best: [11.1, 20.6, 18.6]


[{'method': 'lwr_8_4_0',
  'seed': 0,
  'best_fitness': 11.072690963745117,
  'final_mean_fitness': -24.216934204101562,
  'generations': 300,
  'wall_seconds': 5855.36008644104,
  'history': [{'best_fitness': -2.522608518600464,
    'best_so_far': -2.522608518600464,
    'fitness_variance': 48047.06640625,
    'gen': 0,
    'mean_fitness': -115.14509582519531,
    'nan_count': 83,
    'wall_s': 54.65407943725586},
   {'best_fitness': -0.20266342163085938,
    'best_so_far': -0.20266342163085938,
    'fitness_variance': 28334.552734375,
    'gen': 1,
    'mean_fitness': -91.3458023071289,
    'nan_count': 41,
    'wall_s': 92.03183126449585},
   {'best_fitness': -2.2003965377807617,
    'best_so_far': -0.20266342163085938,
    'fitness_variance': 20077.35546875,
    'gen': 2,
    'mean_fitness': -85.71258544921875,
    'nan_count': 32,
    'wall_s': 125.75309085845947},
   {'best_fitness': 0.41631197929382324,
    'best_so_far': 0.41631197929382324,
    'fitness_variance': 21804.625,
 

#### lwr_8_1_4

In [10]:
# lwr_8_1_4  (input=8, hidden=1, output=4)
# To try a different allocation, edit the three numbers below.
spec = {(256, 27): 8, (256, 256): 1, (8, 256): 4}
run_experiment("lwr_8_1_4", LWREggRoll, spec)

METHOD: lwr_8_1_4  (300 gens, seeds=[0, 1, 2])
  allocation (input,hidden,output) = (8,1,4)
  [lwr_8_1_4 seed=0] exists (best=42.4), skipping.
  [lwr_8_1_4 seed=1] exists (best=75.2), skipping.
  [lwr_8_1_4 seed=2] exists (best=34.0), skipping.

  lwr_8_1_4: mean_best=50.5  std_best=17.8  mean_final=-29.5
  per-seed best: [42.4, 75.2, 34.0]


[{'method': 'lwr_8_1_4',
  'seed': 0,
  'best_fitness': 42.358863830566406,
  'final_mean_fitness': -30.303518295288086,
  'generations': 300,
  'wall_seconds': 21831.848396539688,
  'history': [{'best_fitness': 4.021111011505127,
    'best_so_far': 4.021111011505127,
    'fitness_variance': 101631.78125,
    'gen': 0,
    'mean_fitness': -149.74838256835938,
    'nan_count': 46,
    'wall_s': 115.27893733978271},
   {'best_fitness': 20.962074279785156,
    'best_so_far': 20.962074279785156,
    'fitness_variance': 94897.53125,
    'gen': 1,
    'mean_fitness': -134.94493103027344,
    'nan_count': 51,
    'wall_s': 209.05646324157715},
   {'best_fitness': 9.162035942077637,
    'best_so_far': 20.962074279785156,
    'fitness_variance': 78908.5859375,
    'gen': 2,
    'mean_fitness': -117.64915466308594,
    'nan_count': 28,
    'wall_s': 295.29386615753174},
   {'best_fitness': 42.358863830566406,
    'best_so_far': 42.358863830566406,
    'fitness_variance': 53451.57421875,
    'gen

#### lwr_4_0_2

In [11]:
# lwr_4_0_2  (input=4, hidden=0, output=2)
# To try a different allocation, edit the three numbers below.
spec = {(256, 27): 4, (256, 256): 0, (8, 256): 2}
run_experiment("lwr_4_0_2", LWREggRoll, spec)

METHOD: lwr_4_0_2  (300 gens, seeds=[0, 1, 2])
  allocation (input,hidden,output) = (4,0,2)
  [lwr_4_0_2 seed=0] exists (best=8.1), skipping.
  [lwr_4_0_2 seed=1] exists (best=5.1), skipping.
  [lwr_4_0_2 seed=2] exists (best=11.8), skipping.

  lwr_4_0_2: mean_best=8.3  std_best=2.7  mean_final=-28.1
  per-seed best: [8.1, 5.1, 11.8]


[{'method': 'lwr_4_0_2',
  'seed': 0,
  'best_fitness': 8.139979362487793,
  'final_mean_fitness': -26.962135314941406,
  'generations': 300,
  'wall_seconds': 10117.667420625687,
  'history': [{'gen': 0,
    'mean_fitness': -128.334716796875,
    'best_fitness': 5.25738525390625,
    'best_so_far': 5.25738525390625,
    'fitness_variance': 81469.4453125,
    'nan_count': 70,
    'wall_s': 64.98626661300659},
   {'gen': 1,
    'mean_fitness': -97.509765625,
    'best_fitness': 2.0724306106567383,
    'best_so_far': 5.25738525390625,
    'fitness_variance': 41382.44140625,
    'nan_count': 40,
    'wall_s': 106.19890451431274},
   {'gen': 2,
    'mean_fitness': -88.74766540527344,
    'best_fitness': -0.4107792377471924,
    'best_so_far': 5.25738525390625,
    'fitness_variance': 28734.025390625,
    'nan_count': 36,
    'wall_s': 139.81232905387878},
   {'gen': 3,
    'mean_fitness': -88.0781478881836,
    'best_fitness': 2.150355339050293,
    'best_so_far': 5.25738525390625,
    'fi

#### lwr_4_1_2

In [12]:
# lwr_4_1_2  (input=4, hidden=1, output=2)
# To try a different allocation, edit the three numbers below.
spec = {(256, 27): 4, (256, 256): 1, (8, 256): 2}
run_experiment("lwr_4_1_2", LWREggRoll, spec)

METHOD: lwr_4_1_2  (300 gens, seeds=[0, 1, 2])
  allocation (input,hidden,output) = (4,1,2)
  [lwr_4_1_2 seed=0] exists (best=60.3), skipping.
  [lwr_4_1_2 seed=1] exists (best=35.7), skipping.
  [lwr_4_1_2 seed=2] exists (best=61.0), skipping.

  lwr_4_1_2: mean_best=52.3  std_best=11.8  mean_final=-22.7
  per-seed best: [60.3, 35.7, 61.0]


[{'method': 'lwr_4_1_2',
  'seed': 0,
  'best_fitness': 60.25419616699219,
  'final_mean_fitness': -23.869491577148438,
  'generations': 300,
  'wall_seconds': 8075.6682760715485,
  'history': [{'best_fitness': 60.25419616699219,
    'best_so_far': 60.25419616699219,
    'fitness_variance': 155980.546875,
    'gen': 0,
    'mean_fitness': -175.32455444335938,
    'nan_count': 54,
    'wall_s': 55.47857856750488},
   {'best_fitness': 24.514554977416992,
    'best_so_far': 60.25419616699219,
    'fitness_variance': 103691.546875,
    'gen': 1,
    'mean_fitness': -133.04014587402344,
    'nan_count': 29,
    'wall_s': 93.95797538757324},
   {'best_fitness': 5.507957458496094,
    'best_so_far': 60.25419616699219,
    'fitness_variance': 86498.1328125,
    'gen': 2,
    'mean_fitness': -124.64302825927734,
    'nan_count': 20,
    'wall_s': 127.18917346000671},
   {'best_fitness': 6.750176906585693,
    'best_so_far': 60.25419616699219,
    'fitness_variance': 77530.5859375,
    'gen': 3,

# Metric: Best Fitness

#### lwr_8_4_0

In [14]:
# lwr_8_4_0  (input=8, hidden=4, output=0)
# To try a different allocation, edit the three numbers below.
spec = {(256, 27): 8, (256, 256): 4, (8, 256): 0}
run_experiment("lwr_8_4_0", LWREggRoll, spec)

METHOD: lwr_8_4_0  (300 gens, seeds=[0, 1, 2])
  allocation (input,hidden,output) = (8,4,0)
  [lwr_8_4_0 seed=0] exists (best=2228.7), skipping.
  [lwr_8_4_0 seed=1] exists (best=1367.7), skipping.
  [lwr_8_4_0 seed=2] exists (best=1253.3), skipping.

  lwr_8_4_0: mean_best=1616.6  std_best=435.4  mean_final=398.2
  per-seed best: [2228.7, 1367.7, 1253.3]


[{'method': 'lwr_8_4_0',
  'seed': 0,
  'best_fitness': 2228.718994140625,
  'final_mean_fitness': 524.7291259765625,
  'generations': 300,
  'wall_seconds': 2324.678327560425,
  'history': [{'gen': 0,
    'mean_fitness': -85.85917663574219,
    'best_fitness': 90.22863006591797,
    'best_so_far': 90.22863006591797,
    'fitness_variance': 47648.75,
    'nan_count': 84,
    'wall_s': 31.026213884353638},
   {'gen': 1,
    'mean_fitness': -62.68482971191406,
    'best_fitness': 123.30257415771484,
    'best_so_far': 123.30257415771484,
    'fitness_variance': 34855.609375,
    'nan_count': 63,
    'wall_s': 38.69411516189575},
   {'gen': 2,
    'mean_fitness': -58.63915252685547,
    'best_fitness': 273.2379150390625,
    'best_so_far': 273.2379150390625,
    'fitness_variance': 35425.0859375,
    'nan_count': 65,
    'wall_s': 46.363906145095825},
   {'gen': 3,
    'mean_fitness': -49.61140441894531,
    'best_fitness': 186.059326171875,
    'best_so_far': 273.2379150390625,
    'fitn

#### lwr_8_4_1

In [15]:
# lwr_8_4_1  (input=8, hidden=4, output=1)
# To try a different allocation, edit the three numbers below.
spec = {(256, 27): 8, (256, 256): 4, (8, 256): 1}
run_experiment("lwr_8_4_1", LWREggRoll, spec)

METHOD: lwr_8_4_1  (300 gens, seeds=[0, 1, 2])
  allocation (input,hidden,output) = (8,4,1)
  [lwr_8_4_1 seed=0] exists (best=30.3), skipping.
  [lwr_8_4_1 seed=1] exists (best=21.9), skipping.
  [lwr_8_4_1 seed=2] exists (best=32.6), skipping.

  lwr_8_4_1: mean_best=28.3  std_best=4.6  mean_final=-32.8
  per-seed best: [30.3, 21.9, 32.6]


[{'method': 'lwr_8_4_1',
  'seed': 0,
  'best_fitness': 30.31700325012207,
  'final_mean_fitness': -37.51385498046875,
  'generations': 300,
  'wall_seconds': 2350.738201379776,
  'history': [{'gen': 0,
    'mean_fitness': -155.67108154296875,
    'best_fitness': 30.31700325012207,
    'best_so_far': 30.31700325012207,
    'fitness_variance': 122621.203125,
    'nan_count': 90,
    'wall_s': 32.58876943588257},
   {'gen': 1,
    'mean_fitness': -118.90402221679688,
    'best_fitness': 23.48809814453125,
    'best_so_far': 30.31700325012207,
    'fitness_variance': 69023.78125,
    'nan_count': 59,
    'wall_s': 44.161476135253906},
   {'gen': 2,
    'mean_fitness': -115.93218994140625,
    'best_fitness': 14.373944282531738,
    'best_so_far': 30.31700325012207,
    'fitness_variance': 72099.6171875,
    'nan_count': 51,
    'wall_s': 51.94782209396362},
   {'gen': 3,
    'mean_fitness': -102.78645324707031,
    'best_fitness': 0.1645057201385498,
    'best_so_far': 30.31700325012207,


#### lwr_4_2_0

In [16]:
# lwr_4_2_0  (input=4, hidden=2, output=0)
# To try a different allocation, edit the three numbers below.
spec = {(256, 27): 4, (256, 256): 2, (8, 256): 0}
run_experiment("lwr_4_2_0", LWREggRoll, spec)

METHOD: lwr_4_2_0  (300 gens, seeds=[0, 1, 2])
  allocation (input,hidden,output) = (4,2,0)
  [lwr_4_2_0 seed=0] exists (best=2303.6), skipping.
  [lwr_4_2_0 seed=1] exists (best=1242.5), skipping.
  [lwr_4_2_0 seed=2] exists (best=2071.9), skipping.

  lwr_4_2_0: mean_best=1872.6  std_best=455.5  mean_final=646.3
  per-seed best: [2303.6, 1242.5, 2071.9]


[{'method': 'lwr_4_2_0',
  'seed': 0,
  'best_fitness': 2303.61572265625,
  'final_mean_fitness': 656.8804931640625,
  'generations': 300,
  'wall_seconds': 2328.169801712036,
  'history': [{'gen': 0,
    'mean_fitness': -96.8929443359375,
    'best_fitness': 203.75955200195312,
    'best_so_far': 203.75955200195312,
    'fitness_variance': 59371.8828125,
    'nan_count': 79,
    'wall_s': 32.98307704925537},
   {'gen': 1,
    'mean_fitness': -61.20990753173828,
    'best_fitness': 161.1935577392578,
    'best_so_far': 203.75955200195312,
    'fitness_variance': 32629.962890625,
    'nan_count': 50,
    'wall_s': 40.66099238395691},
   {'gen': 2,
    'mean_fitness': -51.949668884277344,
    'best_fitness': 121.77867889404297,
    'best_so_far': 203.75955200195312,
    'fitness_variance': 26299.43359375,
    'nan_count': 48,
    'wall_s': 48.34269380569458},
   {'gen': 3,
    'mean_fitness': -47.55113983154297,
    'best_fitness': 230.67193603515625,
    'best_so_far': 230.6719360351562

#### lwr_4_2_1

In [17]:
# lwr_4_2_1  (input=4, hidden=2, output=1)
# To try a different allocation, edit the three numbers below.
spec = {(256, 27): 4, (256, 256): 2, (8, 256): 1}
run_experiment("lwr_4_2_1", LWREggRoll, spec)

METHOD: lwr_4_2_1  (300 gens, seeds=[0, 1, 2])
  allocation (input,hidden,output) = (4,2,1)
  [lwr_4_2_1 seed=0] exists (best=34.8), skipping.
  [lwr_4_2_1 seed=1] exists (best=44.6), skipping.
  [lwr_4_2_1 seed=2] exists (best=62.2), skipping.

  lwr_4_2_1: mean_best=47.2  std_best=11.3  mean_final=-26.7
  per-seed best: [34.8, 44.6, 62.2]


[{'method': 'lwr_4_2_1',
  'seed': 0,
  'best_fitness': 34.84602737426758,
  'final_mean_fitness': -28.767868041992188,
  'generations': 300,
  'wall_seconds': 2374.1323766708374,
  'history': [{'gen': 0,
    'mean_fitness': -163.78555297851562,
    'best_fitness': 26.474407196044922,
    'best_so_far': 26.474407196044922,
    'fitness_variance': 149070.921875,
    'nan_count': 66,
    'wall_s': 43.78614068031311},
   {'gen': 1,
    'mean_fitness': -120.29386138916016,
    'best_fitness': 34.84602737426758,
    'best_so_far': 34.84602737426758,
    'fitness_variance': 82388.0625,
    'nan_count': 39,
    'wall_s': 60.012837648391724},
   {'gen': 2,
    'mean_fitness': -118.89258575439453,
    'best_fitness': 12.152162551879883,
    'best_so_far': 34.84602737426758,
    'fitness_variance': 78883.9375,
    'nan_count': 24,
    'wall_s': 67.8190758228302},
   {'gen': 3,
    'mean_fitness': -115.7959976196289,
    'best_fitness': 12.80775260925293,
    'best_so_far': 34.84602737426758,
   